# data processing
refactored to df

## imports

In [13]:
%reload_ext autoreload
%autoreload 2

In [14]:
import sys
import os
from pathlib import Path
import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)

In [15]:
# analysis
from scipy.io import loadmat
from sklearn.decomposition import FastICA
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


In [16]:
# misc
import pickle
from collections import defaultdict

In [17]:
# task
from env_config import Config
from firefly_task import ffacc_real
# from monkey_functions import *
# from InverseFuncs import *
from stable_baselines3 import TD3
import torch


In [18]:
from neural_plot_ult import *
import time
tic=time.time()
import warnings
warnings.filterwarnings('ignore')

In [19]:
# run this once per notebook session  —  ideally in the first cell
from IPython import get_ipython

def notify_exc(shell, etype, evalue, tb, tb_offset=None):
    # 1. push notification
    summary = f"{etype.__name__}: {evalue}"
    notify(msg=summary, title="Python Error")

    # 2. let IPython show its normal traceback
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return None            # ← must be None or a list of strings

get_ipython().set_custom_exc((Exception,), notify_exc)


# Pre IRC

convert the mat data file (with neural data) into (states, actions, tasks) for IRC.


## prepare

In [20]:
# const
bin_size = 1  # how many bin of DT
# num_bins = 24  # how many bins to use. use 2.4 s and discard the long trials.
# monkey_height = 10
DT = 0.006  # DT for raw data
# reward_boundary = 65
areas = ['PPC', 'PFC', 'MST','VIP']
worldscale = 200

m = 'm53'
folder = 'mat'
dens = [0.0001, 0.0005, 0.001,  0.005]

locals().update({m: {}})
figure_path = resdir/'figures'
# datapaths = [i for i in Pa1th(resdir/'mat_ruiyi').glob(f'{m}*.mat')]
datapaths = [i for i in Path(resdir/folder).glob(f'{m}*.mat')]
session=sorted([int((a.stem).split('s')[1]) for a in datapaths])
# session,datapaths

## from raw data file: task relavent variables and neural

In [21]:
# load raw data
df=pd.DataFrame() 
for idx, datapath in enumerate(datapaths):
    if datapath.stem[-1].isalpha():
        continue
    data = loadmat(datapath)
    eval(m)[datapath.stem] = data

    # df
    sessdf=pd.DataFrame() 
    sessdf['trial']=np.arange((len(data['trials_behv'][0])))
    sessdf['session']=int(datapath.stem[(datapath.stem).find('s')+1:])
    df=pd.concat([df, sessdf])

In [22]:
sessdata = defaultdict(list)

for key, data in eval(m).items():
    sess = int(key.split('s')[1])
    if key[-1].isalpha():
        continue
    
    trials_behv = data['trials_behv'][0]
    trials_units = data['units'][0]
    units_area = np.array([v[0] for v in trials_units['brain_area']])
    trials_error = []
    trials_error_sign = []
    trials_target_angle = []
    trials_target_distance = []

    print(len(trials_behv), len(trials_units[0]['trials'][0]), key)
    for trial_idx, trial_behv in enumerate(trials_behv):
        trial_ts = trial_behv['continuous']['ts'][0][0].reshape(-1)
        t_mask = (trial_ts > 0) & (
            ~np.isnan(trial_behv['continuous']['ymp'][0][0].reshape(-1)))
        t_mask &= trial_ts < trial_behv['events']['t_stop'][0][0].reshape(-1)
        if t_mask.sum() > 0:
            # remove the first data point to avoid downsample error
            t_mask[np.where(t_mask == True)[0][0]] = False

        # task varaibles from data
        mx = trial_behv['continuous']['xmp'][0][0][t_mask].reshape(-1)
        my = trial_behv['continuous']['ymp'][0][0][t_mask].reshape(-1)
        fx = trial_behv['continuous']['xfp'][0][0][t_mask].reshape(-1)
        fy = trial_behv['continuous']['yfp'][0][0][t_mask].reshape(-1)
        eye_hor_theta = trial_behv['continuous']['yre'][0][0][t_mask].reshape(-1) # use yle for left eye.
        eye_ver_theta = trial_behv['continuous']['zre'][0][0][t_mask].reshape(-1)
        mv = trial_behv['continuous']['v'][0][0][t_mask].reshape(-1)
        mw = trial_behv['continuous']['w'][0][0][t_mask].reshape(-1)

        # some adjustment for screen distance
        sx = np.ones_like(fx)
        sy = np.ones_like(fy)
        if my.size > 0:
            fx = np.ones_like(fx) * fx[0]
            fy = np.ones_like(fy) * fy[0]
            sx *= mx[-1]
            sy *= my[-1]
            my = my + 30
            fy = fy + 30
            sy = sy + 30

        # some thing coudl be removed from there.
        dx = fx - mx; dy = fy - my
        rel_dist = np.sqrt(dx**2 + dy**2); rel_ang = np.rad2deg(np.arctan2(dy, dx))
        rel_dist_stop = np.sqrt((sx - mx)**2 + (sy - my)**2)
        abs_dist = np.sqrt(mx**2 + my**2); abs_ang = np.rad2deg(np.arctan2(my, mx))
        heading = np.deg2rad(np.cumsum(mw*-1) * DT + 90)
        body_x, body_y = mx.reshape(-1), my.reshape(-1)

        # skip bad trial
        if t_mask.sum() * DT > 3.5 or t_mask.sum() * DT < 0.6 or mv.max() < 50 or \
                abs_dist[-1] < np.sqrt(fx**2 + fy**2)[-1] * 0.3:
            continue

        # errors
        if my.size > 0:
            trials_error.append(rel_dist[-1])
            trials_error_sign.append(rel_dist[-1])
            trials_target_angle.append(
                np.rad2deg(np.arctan2(fy, fx))[-1] - 90)
            trials_target_distance.append(np.sqrt(fx**2 + fy**2)[-1])
            d1 = np.sqrt(fx**2 + fy**2)
            r1 = (fx**2 + fy**2) / (2*fx)
            radian1 = 2 * r1 * np.arcsin(d1 / (2 * r1))
            d2 = np.sqrt(mx**2 + my**2)
            r2 = (mx**2 + my**2) / (2*mx + 1e-8)
            radian2 = 2 * r2 * np.arcsin(d2 / (2 * r2 + 1e-8))
            sign = np.ones_like(rel_dist)
            sign[radian2 < radian1] = -1
            rel_dist = sign * rel_dist
            trials_error_sign[-1] = rel_dist[-1]
        else:
            trials_error.append(np.nan)
            trials_error_sign.append(np.nan)
            trials_target_angle.append(np.nan)
            trials_target_distance.append(np.nan)

        heading = -np.deg2rad(np.cumsum(mw) * DT - 90) # 90 is up
        egox, egoy=world_to_egocentric(fx, fy, mx, my, heading)
        latent_ff_hori, latent_ff_vert=convert_egolocation_to_angle(egox, egoy)
      
        # df
        # basis
        sessdata['session'].append(int(sess))
        sessdata['trial'].append(trial_idx)
        sessdata['fullon'].append(trial_behv['logical']['firefly_fullON'][0][0][0][0])
        sessdata['ptb'].append(trial_behv['logical']['ptb'][0][0][0][0])
        sessdata['density'].append((trial_behv['prs'][0][0]['floordensity'].item()))
        
        # world coord
        # sessdata['rel_dist'].append(rel_dist)
        # sessdata['rel_ang'].append(rel_ang)
        sessdata['heading'].append(heading) # 90 degree is up
        sessdata['fx'].append(fx)
        sessdata['fy'].append(fy)
        sessdata['mx'].append(mx)
        sessdata['my'].append(my)
        sessdata['mv'].append(mv)
        sessdata['mw'].append(mw)

        # ego coord
        sessdata['egofx'].append(egox) # mk heading is y axis.
        sessdata['egofy'].append(egoy)
        # sessdata['abs_dist'].append(abs_dist)
        # sessdata['abs_ang'].append(abs_ang)

        # eye coord
        sessdata['eye_hori'].append(eye_hor_theta)
        sessdata['eye_vert'].append(eye_ver_theta)
        sessdata['ff_hori'].append(latent_ff_hori.reshape(-1))
        sessdata['ff_vert'].append(latent_ff_vert.reshape(-1))

        # neural
        activities = []  # activities for all neurons for 1 trial. shape: ts, neurons
        for trials_unit in trials_units:
            fire_ts = trials_unit['trials'][0][trial_idx][0].reshape(-1)
            if fire_ts.size > 0 and fire_ts[-1] >= trial_ts[-1]:
                fire_ts = fire_ts[:-1]
            activity = np.zeros_like(trial_ts)
            bin_indices = np.digitize(fire_ts, trial_ts)
            unique_bins, bin_counts = np.unique(
                bin_indices, return_counts=True)
            activity[unique_bins] = bin_counts
            activities.append(activity)

        activities = np.vstack(activities).T   # time * unit
        activities = activities[t_mask]
        activities = gaussian_filter1d(
            activities, sigma=4, axis=0)  # neural for each trial
        activity = downsample(activities, bin_size=bin_size)
        activity_var = downsample_variance(activities, bin_size=bin_size)
        
        for area in areas:
            area_mask = [v in area for v in units_area]
            if sum(area_mask) == 0:
                activity_ = np.nan
            else:
                activity_ = activity[:, area_mask]  # area activity
                activity_var_=activity_var[:,area_mask]
            sessdata[area].append(activity_)
            sessdata[f'{area}_var'].append(activity_var_)

    sessdata['error'] += (trials_error)
    sessdata['error_sign'] += (trials_error_sign)
    sessdata['target_angle'] += (trials_target_angle)
    sessdata['target_distance'] += (trials_target_distance)

tmp = pd.DataFrame(sessdata)
df = pd.merge(df, tmp, on=['trial', 'session'], how='inner')

1500 1500 m53s116
1260 1260 m53s100
1629 1629 m53s114
1500 1500 m53s111
1511 1511 m53s39
1117 1117 m53s106
1500 1500 m53s48
1751 1751 m53s49
1500 1500 m53s98
1500 1500 m53s95
1501 1501 m53s42
1502 1502 m53s43
1567 1567 m53s41
1700 1700 m53s40
846 846 m53s83
1522 1522 m53s44
1500 1500 m53s50
1813 1813 m53s51
1022 1022 m53s92
1508 1508 m53s86
449 449 m53s90
1501 1501 m53s47
2203 2203 m53s46
1405 1405 m53s35
1500 1500 m53s123
1591 1591 m53s34
1500 1500 m53s36
1501 1501 m53s109
1522 1522 m53s134
1505 1505 m53s37
1100 1100 m53s32
1410 1410 m53s133
1500 1500 m53s127
1500 1500 m53s132
1501 1501 m53s31


In [23]:
df.columns

Index(['trial', 'session', 'fullon', 'ptb', 'density', 'heading', 'fx', 'fy',
       'mx', 'my', 'mv', 'mw', 'egofx', 'egofy', 'eye_hori', 'eye_vert',
       'ff_hori', 'ff_vert', 'PPC', 'PPC_var', 'PFC', 'PFC_var', 'MST',
       'MST_var', 'VIP', 'VIP_var', 'error', 'error_sign', 'target_angle',
       'target_distance'],
      dtype='object')

## IRC input data (state, action, task)

In [24]:
def task2irc(fx, fy, mx, my, worldscale=200):

    # Calculate relative position
    task_x = (fx - mx).astype('float32')
    task_y = (fy - my).astype('float32')
    
    # Apply scaling and return in the format [y_coord, x_coord]
    return [task_y/worldscale, task_x/worldscale]


In [25]:
sessdata=defaultdict(list)

for sess in session:
    
    states, actions, tasks=[],[],[]
    sessdf=df[df.session==sess]
    trial_idces=sessdf.trial

    for trial_idx in trial_idces:
        trialdf=sessdf[sessdf.trial==trial_idx]
        trialdata=trialdf.iloc[0]
        # task
        taskx = (trialdata.fx[0] - trialdata.mx[0]).astype('float32'); tasky = (trialdata.fy[0] - trialdata.my[0]).astype('float32')
        tasks.append([tasky/worldscale,taskx/worldscale])
  
        trialaction=np.stack([trialdata.mv,trialdata.mw]).T
        trialaction[:,0]=trialaction[:,0]/worldscale # v need reduce scale
        trialaction[:,1]=trialaction[:,1]/180*pi
        actions.append(trialaction.astype('float32'))

        # states from run the actions
        px, py, heading, v, w = 0,0,0,0,0
        log=[]
        for a in trialaction:
            px, py, heading, v, w=state_step2(px, py, heading, v, w, a, dt=DT,userad=True)
            log.append([px, py, heading, v, w])
        px, py, heading, v, w=state_step2(px, py, heading, v, w, a, dt=DT,userad=True)
        log.append([px, py, heading, v, w])
        trialstates=np.array(log)[1:]
        
        states.append(trialstates.astype('float32'))

        sessdata['session'].append(sess)
        sessdata['trial'].append(trial_idx)
        sessdata['state'].append(trialstates.astype('float32'))  
        sessdata['action'].append(trialaction.astype('float32'))  
        sessdata['task'].append([tasky/worldscale,taskx/worldscale])  


In [26]:

tmp=pd.DataFrame(sessdata)
df = pd.merge(df, tmp, on=['trial','session'], how='inner')

## Compute belief 

In [80]:
# model estimated likelihood (negative log likelihood)
torch.manual_seed(42)
arg = Config()

env = ffacc_real.FireFlyPaper(arg)
env.debug=True
phi = torch.tensor([[0.5],
                    [pi/2],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.13],
                    [0.001],
                    [0.001],
                    [0.001],
                    [0.001],
                    ])

agent_ = TD3.load(workdir/'trained_agent/paper')
agent = agent_.actor.mu.cpu()

tensor([0.])


In [81]:
env.dt=DT

In [97]:
thetas={}
for idensity in range(4):
    # datapath = Path(resdir/f'{folder}/preirc_den_{idensity}')
    # savename = datapath.parent/(f'{m}_{idensity}'+datapath.name)
    # invfile=savename
    invfile=resdir/'shro_0430'
    with open(invfile, 'rb') as f:
        logs = pickle.load(f)
    ll=[np.mean(log[-1]) for log in logs]
    bestind=np.argmin(ll)
    res=logs[bestind]
    finaltheta, finalcov, err = res
    # print('finaltheta', finaltheta._mean)
    # print('finalcov', finalcov)
    # print('likelihood', err)
    finaltheta = finaltheta._mean
    finnatheta=np.array(finaltheta)
    finaltheta=np.concatenate([finaltheta[:6],np.array([0.05]), finaltheta[6:]])
    finaltheta=finaltheta.astype('float32')
    # finaltheta[0]=1
    # finaltheta[1]=1.3
    # finaltheta[1]=0.5
    # finaltheta[1]=0.2
    finaltheta=torch.tensor(finaltheta).reshape(-1)
    thetas[idensity]=finaltheta


## Compute likelihood
todo, need to do this trial by trial to take account of density.

In [98]:
# # df likelihood
# today='0507'
# skipll=not(date.today().strftime("%m%d") == today)

# def lltrial(state, action, task, finaltheta, samples=5):
#     with torch.no_grad():
#         return monkeyloss_(agent, action, np.array(task).reshape(1,-1), phi, finaltheta, env, action_var=0.01, num_iteration=1, states=state, samples=samples, gpu=False).item()

# likelihood_df=defaultdict(list)

# if not skipll: 
#     # compute new ll and save with today date
#     for sess in session:
#         sessdf=df[df.session==sess]
#         state, action, task = df.state.to_list(), df.action.to_list(),df.task.to_list()
#         trial_idces=sessdf.trial
#         for trial_idx in trial_idces:
#             trialdf=sessdf[sessdf.trial==trial_idx]
#             trialdata=trialdf.iloc[0]
#             state, action, task = trialdata.state, trialdata.action, trialdata.task
#             trial_likelihood=lltrial([state], [action], [task], finaltheta) 

#             likelihood_df['session'].append(sess)
#             likelihood_df['trial'].append(trial_idx)
#             likelihood_df['likelihood'].append(trial_likelihood)
        
#     today = date.today().strftime("%m%d") # mm/dd
#     with open(resdir/f'{folder}/{m}_irc_ll_{today}','wb+') as f:
#         pickle.dump(likelihood_df,f)
#     notify('compute likelihood complete')
#     print(f'computed likelihood saved to {folder}/irc_ll_{today}')

# else: # use the given day
#     print(f'use computed likelihood from day {today}')
#     with open(resdir/f'{folder}/irc_ll_{today}','rb') as f:
#             likelihood_df=pickle.load(f)

# tmp=pd.DataFrame(likelihood_df)
# df = pd.merge(df, tmp, on=['trial','session'], how='inner')

In [99]:
denslookup={0.0001:0, 0.0005:1, 0.001:2,  0.005:3}
thisdensity=trialdf.density.item()
denslookup[thisdensity],thisdensity

(0, 0.0001)

In [100]:
# df belief
today='0507'
skipblief=not(date.today().strftime("%m%d") == today)

def lltrial(state, action, task, finaltheta, samples=5):
    with torch.no_grad():
        return monkeyloss_(agent, action, np.array(task).reshape(1,-1), phi, finaltheta, env, action_var=0.01, num_iteration=1, states=state, samples=samples, gpu=False).item()

belief_df=defaultdict(list)

if not skipblief:
    for sess in session:
        print(sess)
        sessdf=df[df.session==sess]
        state, action, task = df.state.to_list(), df.action.to_list(),df.task.to_list()
        trial_idces=sessdf.trial
        for trial_idx in trial_idces:
            trialdf=sessdf[sessdf.trial==trial_idx]
            trialdata=trialdf.iloc[0]
            state, action, task = trialdata.state, trialdata.action, trialdata.task
            thisdensity=trialdf.density.item()
            theta=thetas[denslookup[thisdensity]]

            _, _, ep_belief, ep_rawcov = run_trials(agent=agent, 
                                                   env=env, phi=phi, theta=theta,          
                                                   task=task, ntrials=1,
                                                    pert=None, given_obs=None, return_belief=True, given_action=action, given_state=state)
            # trial info
            belief_df['session'].append(sess)
            belief_df['trial'].append(trial_idx)

            # belief
            if len(state)<5: # 
                belief_df['belief'].append(np.nan)
                belief_df['rawcov'].append(np.nan)
            else:
                init=torch.tensor(state[0]).reshape(-1,1)
                trial_belief=(ep_belief[0]-ep_belief[0][0]+init)
                belief_df['belief'].append(np.array(trial_belief)[:,:,0])
                belief_df['rawcov'].append(np.array(ep_rawcov[0]))
        
    notify('all done')
    today = date.today().strftime("%m%d") # mm/dd
    with open(resdir/f'{folder}/{m}_irc_belief_{today}','wb+') as f:
        pickle.dump(belief_df,f)
    notify('compute belief complete')
    print(f'computed belief saved to {folder}/irc_belief_{today}')

else: # use the given day
    print(f'use computed belief from day {today}')
    with open(resdir/f'{folder}/irc_belief_{today}','rb') as f:
            belief_df=pickle.load(f)
tmp=pd.DataFrame(belief_df)
tmp=tmp.rename(columns={'cov': 'rawcov'})
df = pd.merge(df, tmp, on=['trial','session'], how='inner')

31
32
34
35
36
37
39
40
41
42
43
44
46
47
48
49
50
51
83
86
90
92
95
98
100
106
109
111
114
116
123
127
132
133
134
computed belief saved to mat/irc_belief_0508


In [101]:
del env

In [102]:
# unpack the belief state
df['bmx']=df.apply(lambda x:x.belief[:,1]*worldscale, axis=1)
df['bmy']=df.apply(lambda x:x.belief[:,0]*worldscale, axis=1)
df['belief_heading']=df.apply(lambda x: x.belief[:,2]*180/pi, axis=1)
df['timer']=df.apply(lambda x: np.arange(len(x.mx)), axis=1)
df['countdown']=df.apply(lambda x: np.flip(-np.arange(len(x.mx)), axis=0), axis=1)

In [103]:
sessdata=defaultdict(list)

for sess in session:
    sessdf=df[df.session==sess]
    states, actions, tasks = sessdf.state.to_list(), sessdf.action.to_list(),sessdf.task.to_list()
    beliefs,rawcovs=sessdf['belief'].to_list(),sessdf['rawcov'].to_list()

    sess_latentff_hori, sess_latentff_vert = [], []
    for ep_beliefs, ep_rawcovs, task in zip(beliefs, rawcovs, tasks): # process for each trial
        mx, my, heading,  mv, mw = zip(*ep_beliefs)
      
        body_x, body_y = np.asarray(my).reshape(-1).astype('float') * \
            worldscale, np.asarray(mx).reshape(-1).astype('float')*worldscale

        fx, fy = task[1]*worldscale, task[0]*worldscale
        
        heading = -np.deg2rad(np.cumsum(mw) * DT - 90) # 90 is up
        egox, egoy=world_to_egocentric(fx, fy, mx, my, heading)
        latent_ff_hori, latent_ff_vert=convert_egolocation_to_angle(egox, egoy)
      
        
        sess_latentff_hori.append(latent_ff_hori)
        sess_latentff_vert.append(latent_ff_vert)
    sessdata['belief_ff_hori']+=[a.reshape(-1) for a in sess_latentff_hori]
    sessdata['belief_ff_vert']+=[a.reshape(-1) for a in sess_latentff_vert]
    


df['belief_ff_hori']=sessdata['belief_ff_hori']
df['belief_ff_vert']=sessdata['belief_ff_vert']

In [104]:
# angle from start
def get_angle_from_start(row):
    return np.arctan2((np.array(row.my)), (np.array(row.mx)))

df['angle_from_start']=df.apply(get_angle_from_start, axis=1)

def get_belief_angle_from_start(row):
    return np.arctan2((np.array(row.bmy)), (np.array(row.bmx)))

df['belief_angle_from_start']=df.apply(get_angle_from_start, axis=1)


# rotate belief cov df. this is the cov under foot. 
def fun(trialdf):
    if trialdf.fullon==1: # always on target, assign zero uncertainty
        return np.ones_like(trialdf['rawcov'][:,:2,:2])*1e-6
    cov=trialdf['rawcov'][:,:2,:2]
    belief_heading=trialdf.belief_heading
    rotdegree=belief_heading+180
    relativeposcov=[]
    for degree, thiscov in zip(rotdegree, cov):
        R=np.array([[np.cos(-degree/180*pi),-np.sin(-degree/180*pi)],[np.sin(-degree/180*pi),np.cos(-degree/180*pi)]])
        relativeposcov.append(R.T@thiscov[:2,:2]@R)
    relativeposcov=np.stack(relativeposcov)*worldscale*worldscale
    return relativeposcov

df['relcov']=df.apply(fun, axis=1) # the relative cov is both for self location and target center, since its rolated.

## the varialbes we have:

In [105]:
# with neuron variance as PPC_var
date='0507'
df.to_pickle(resdir/f'{date}_m53df.pkl')
notify(f'saved: {date}_m53df.pkl')
print(f'saved: {date}_m53df.pkl')

saved: 0507_m53df.pkl
